In [ ]:
import os, sys
from pathlib import Path
_cwd = Path.cwd()
_root = next((p for p in [_cwd] + list(_cwd.parents)
              if (p / 'requirements.txt').exists() and (p / 'src').is_dir()), None)
assert _root, f'Could not find project root from {_cwd}'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')

# 01 — Exploratory Data Analysis

This notebook explores the **synthetic fixture panel** (30 days × 48 settlement periods).  Run against real data by replacing `load_fixture_panel()` with `load_fixture_panel()` pointed at `data/processed/panel.parquet` (see `data/README.md`).

We visualise:
1. Price over time
2. Daily price profile (mean by SP-of-day)
3. Price vs demand scatter
4. Price distribution + spike note
5. Generation mix stacked-area

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe for nbconvert
import matplotlib.pyplot as plt

from src.build.fixtures import load_fixture_panel

# Use live panel if it exists, otherwise fall back to fixture
_LIVE_PATH = Path('data/processed/panel.parquet')
if _LIVE_PATH.exists():
    print('Loading LIVE panel from', _LIVE_PATH)
    import pandas as pd
    bundle_raw = pd.read_parquet(_LIVE_PATH)
    print('Live panel shape:', bundle_raw.shape)
    USE_LIVE = True
else:
    print('Using FIXTURE panel — see data/README.md for live data')
    USE_LIVE = False

bundle = load_fixture_panel()
print('Fixture bundle loaded.')
print('  target         :', bundle.target.n_timesteps, 'steps')
print('  past_covariates:', bundle.past_covariates.components.tolist())
print('  future_covar.  :', bundle.future_covariates.components.tolist())

In [ ]:
# ── Extract raw DataFrames for plotting ─────────────────────────────────────
price_s = bundle.target.to_series()          # price
past_df  = bundle.past_covariates.to_dataframe()   # demand + gen_*

# Add sp_of_day from future covariates for daily profile
fut_df = bundle.future_covariates.to_dataframe()
price_df = price_s.rename('price').to_frame()
price_df['sp_of_day'] = fut_df['sp_of_day'].values

## 1. Price over time

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(price_s.index, price_s.values, lw=0.8, color='royalblue')
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Price (£/MWh)')
ax.set_title('GB Half-hourly Settlement Price — fixture panel')
ax.grid(True, alpha=0.3)
fig.tight_layout()
_out = Path('reports/figures/01_price_timeseries.png')
_out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(_out, dpi=120)
plt.show()
print('Saved:', _out)

## 2. Daily price profile (mean by SP-of-day)

In [ ]:
profile = price_df.groupby('sp_of_day')['price'].agg(['mean', 'std'])
profile.index = profile.index.astype(int)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(profile.index, profile['mean'], color='royalblue', lw=1.5, label='Mean price')
ax.fill_between(
    profile.index,
    profile['mean'] - profile['std'],
    profile['mean'] + profile['std'],
    alpha=0.25, color='royalblue', label='±1 SD'
)
ax.set_xlabel('Settlement period of day (1 = 00:00–00:30 UTC)')
ax.set_ylabel('Price (£/MWh)')
ax.set_title('Daily price profile — fixture panel')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('reports/figures/01_daily_profile.png', dpi=120)
plt.show()
print('Saved: reports/figures/01_daily_profile.png')

## 3. Price vs demand scatter

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(past_df['demand_indo'], price_s.values, s=4, alpha=0.3, color='steelblue')
ax.set_xlabel('Demand — INDemand Outturn (MW)')
ax.set_ylabel('Price (£/MWh)')
ax.set_title('Price vs Demand')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 4. Price distribution + spike note

> **Price spikes**: The UK electricity spot market can exhibit large positive spikes (scarcity pricing, system stress events).  The fixture panel includes 12 synthetic spikes +30–100 £/MWh.  In production data, spikes can exceed several thousand £/MWh.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].hist(price_s.values, bins=50, color='royalblue', edgecolor='white', lw=0.3)
axes[0].set_xlabel('Price (£/MWh)')
axes[0].set_ylabel('Count')
axes[0].set_title('Price distribution (full range)')
axes[0].grid(True, alpha=0.3)

# Zoom in to 5–130 £/MWh to show the bulk + minor spikes
bulk = price_s[price_s < 130]
axes[1].hist(bulk.values, bins=50, color='steelblue', edgecolor='white', lw=0.3)
axes[1].set_xlabel('Price (£/MWh)')
axes[1].set_title('Price distribution (bulk, <130 £/MWh)')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Settlement price distribution — fixture panel', y=1.02)
fig.tight_layout()
plt.show()
print(f'p99 price: {price_s.quantile(0.99):.1f}  max: {price_s.max():.1f}')

## 5. Generation mix stacked-area

In [ ]:
gen_cols = [c for c in past_df.columns if c.startswith('gen_')]
gen_df   = past_df[gen_cols].copy()
# Normalise column labels for the legend
gen_df.columns = [c.replace('gen_', '') for c in gen_cols]

fig, ax = plt.subplots(figsize=(13, 4))
ax.stackplot(
    gen_df.index,
    [gen_df[c] for c in gen_df.columns],
    labels=gen_df.columns,
    alpha=0.8,
)
ax.legend(loc='upper left', ncol=3, fontsize=8)
ax.set_ylabel('Generation (MW)')
ax.set_xlabel('Timestamp (UTC)')
ax.set_title('Generation mix — fixture panel')
ax.grid(True, alpha=0.2)
fig.tight_layout()
fig.savefig('reports/figures/01_gen_mix.png', dpi=120)
plt.show()
print('Saved: reports/figures/01_gen_mix.png')